In [53]:
#%pip install -U langchain-cohere cohere

In [54]:
#%pip install -U langgraph langchain langchain-openai pydantic pymupdf

In [55]:
import os
import getpass

os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API Key: ")

Enter Cohere API Key: ··········


In [56]:
import sys

print("Python:", sys.version)

import langgraph
import langchain
import langchain_openai
import pydantic
import fitz


print("LangChain:", langchain.__version__)
print("LangChain OpenAI: OK")
print("Pydantic:", pydantic.__version__)
print("PyMuPDF: OK")

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
LangChain: 1.3.18
LangChain OpenAI: OK
Pydantic: 2.13.5
PyMuPDF: OK


In [57]:
pip show langgraph

Name: langgraph
Version: 1.2.11
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain


In [58]:
# ============================================================
# RFP EVALUATION SYSTEM
# LANGGRAPH VERSION
# NO STREAMLIT
#
# Architecture:
#
#   LangGraph Orchestrator
#          |
#          +--> Document Tool
#          |
#          +--> Evaluation Agent
#          |
#          +--> Validation Tool
#          |
#          +--> Deterministic Ranking Tool
#          |
#          +--> SQLite Persistence
#
# ============================================================

import os
import json
import math
import sqlite3

from datetime import datetime
from typing import TypedDict, List, Dict, Any

import fitz

from pydantic import BaseModel, Field

from langchain_core.tools import tool
#from langchain_openai import ChatOpenAI
from langchain_cohere import ChatCohere

from langgraph.graph import StateGraph, START, END


# ============================================================
# CONFIGURATION
# ============================================================

DB_PATH = "rfp_evaluator.db"

MODEL_NAME = "command-a-plus-05-2026"


# ============================================================
# DATABASE
# ============================================================

def get_connection():

    return sqlite3.connect(DB_PATH)


def table_exists(cursor, table_name):

    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        AND name = ?
    """, (table_name,))

    return cursor.fetchone() is not None


def initialize_database():

    conn = get_connection()
    cursor = conn.cursor()

    # --------------------------------------------------------
    # evaluation_criteria
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS evaluation_criteria (

            criterion_id INTEGER PRIMARY KEY AUTOINCREMENT,

            name TEXT NOT NULL,

            description TEXT NOT NULL,

            weight REAL NOT NULL,

            max_score REAL NOT NULL DEFAULT 10,

            is_active INTEGER NOT NULL DEFAULT 1
        )
    """)

    # --------------------------------------------------------
    # rfp_runs
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS rfp_runs (

            rfp_run_id INTEGER PRIMARY KEY AUTOINCREMENT,

            created_at TEXT NOT NULL,

            status TEXT NOT NULL
        )
    """)

    # --------------------------------------------------------
    # rfp_run_criteria
    #
    # Snapshot of criteria used for each run.
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS rfp_run_criteria (

            run_criterion_id INTEGER PRIMARY KEY AUTOINCREMENT,

            rfp_run_id INTEGER NOT NULL,

            criterion_id INTEGER NOT NULL,

            name TEXT NOT NULL,

            description TEXT NOT NULL,

            weight REAL NOT NULL,

            max_score REAL NOT NULL
        )
    """)

    # --------------------------------------------------------
    # rfp_suppliers
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS rfp_suppliers (

            supplier_id INTEGER PRIMARY KEY AUTOINCREMENT,

            rfp_run_id INTEGER NOT NULL,

            supplier_name TEXT NOT NULL,

            submission_date TEXT NOT NULL,

            experience_rating REAL NOT NULL,

            absolute_score REAL,

            ppi REAL,

            final_rank INTEGER,

            result_json TEXT
        )
    """)

    # --------------------------------------------------------
    # evaluation_results
    # --------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS evaluation_results (

            evaluation_id INTEGER PRIMARY KEY AUTOINCREMENT,

            supplier_id INTEGER NOT NULL,

            criterion_id INTEGER NOT NULL,

            raw_score REAL,

            normalized_score REAL,

            justification TEXT,

            evidence_json TEXT,

            warnings_json TEXT,

            weighted_score REAL,

            benchmark_score REAL,

            criterion_gap REAL,

            relative_percentage REAL
        )
    """)

    # --------------------------------------------------------
    # Seed criteria ONLY if table is empty
    #
    # Existing criteria will NOT be deleted.
    # --------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM evaluation_criteria
    """)

    criteria_count = cursor.fetchone()[0]

    if criteria_count == 0:

        criteria = [

            (
                "Technical Capability",
                "Architecture, integrations, scalability, technical fit",
                0.30,
                10
            ),

            (
                "Implementation Plan",
                "Timeline, milestones, staffing, risk plan",
                0.20,
                10
            ),

            (
                "Commercial Value",
                "Pricing clarity, total cost, assumptions",
                0.20,
                10
            ),

            (
                "Security & Compliance",
                "Controls, certifications, privacy, auditability",
                0.20,
                10
            ),

            (
                "Support & Experience",
                "Support model, similar projects, references",
                0.10,
                10
            )
        ]

        cursor.executemany("""
            INSERT INTO evaluation_criteria
            (
                name,
                description,
                weight,
                max_score
            )
            VALUES (?, ?, ?, ?)
        """, criteria)

        print(
            "Default evaluation criteria inserted."
        )

    else:

        print(
            "Existing evaluation criteria preserved."
        )

    conn.commit()
    conn.close()


In [59]:
# ============================================================
# LOAD ACTIVE CRITERIA
# ============================================================

def load_active_criteria():

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            criterion_id,
            name,
            description,
            weight,
            max_score
        FROM evaluation_criteria
        WHERE is_active = 1
        ORDER BY criterion_id
    """)

    rows = cursor.fetchall()

    conn.close()

    criteria = []

    for row in rows:

        criteria.append({

            "criterion_id": row[0],

            "name": row[1],

            "description": row[2],

            "weight": float(row[3]),

            "max_score": float(row[4])
        })

    return criteria



In [60]:
# ============================================================
# VALIDATE ACTIVE CRITERIA
# ============================================================

def validate_criteria(criteria):

    if not criteria:

        raise ValueError(
            "No active evaluation criteria found."
        )

    total_weight = sum(
        c["weight"]
        for c in criteria
    )

    if not math.isclose(
        total_weight,
        1.0,
        abs_tol=1e-9
    ):

        raise ValueError(
            f"Active criteria weights must total 100%. "
            f"Current total = "
            f"{total_weight * 100:.2f}%"
        )

    for criterion in criteria:

        if criterion["weight"] < 0:

            raise ValueError(
                f"Negative weight for "
                f"{criterion['name']}"
            )

        if criterion["max_score"] <= 0:

            raise ValueError(
                f"Invalid max score for "
                f"{criterion['name']}"
            )


In [61]:
# ============================================================
# TOOL 1
# DOCUMENT TOOL
# ============================================================

@tool
def extract_pdf_text(pdf_path: str) -> str:
    """
    Document Tool.

    Extract clean text from a supplier PDF.

    This tool does NOT evaluate the document.
    """

    if not os.path.exists(pdf_path):

        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(
        document,
        start=1
    ):

        text = page.get_text("text")

        if text and text.strip():

            pages.append(
                f"\n--- PAGE {page_number} ---\n"
                f"{text.strip()}"
            )

    document.close()

    if not pages:

        raise ValueError(
            "PDF contains no readable text."
        )

    return "\n".join(pages)


In [62]:
# ============================================================
# PYDANTIC STRUCTURED OUTPUT
# ============================================================

class CriterionEvaluation(BaseModel):

    criterion_id: int = Field(
        description="ID of evaluation criterion"
    )

    score: float = Field(
        description="Score assigned to proposal"
    )

    justification: str = Field(
        description="Explanation for the score"
    )

    evidence: List[str] = Field(
        description="Evidence found in proposal"
    )


class SupplierEvaluation(BaseModel):

    criteria: List[CriterionEvaluation]


# ============================================================
# AGENT
# EVALUATION AGENT
# ============================================================

class EvaluationAgent:
    """
    LLM Evaluation Agent.

    The LLM is ONLY responsible for judging
    the supplier proposal.

    It does NOT:
        - calculate weighted scores
        - calculate benchmarks
        - calculate PPI
        - rank suppliers
        - apply tie-break rules
    """

    def __init__(self, model_name=MODEL_NAME):

        self.llm = ChatCohere(

            model=model_name,

            temperature=0,

            cohere_api_key=os.environ[
                "COHERE_API_KEY"
            ]
        )

        self.structured_llm = (
            self.llm.with_structured_output(
                SupplierEvaluation
            )
        )

    def evaluate(
        self,
        proposal_text,
        criteria
    ):

        criteria_text = "\n\n".join(

            [
                f"""
Criterion ID: {criterion['criterion_id']}

Criterion:
{criterion['name']}

What to inspect:
{criterion['description']}

Maximum Score:
{criterion['max_score']}
"""
                for criterion in criteria
            ]
        )

        prompt = f"""
You are an RFP Evaluation Agent.

Evaluate ONE supplier proposal against
the active evaluation criteria.

Your role is ONLY to judge the content
of the proposal.

You MUST NOT:

- calculate weighted scores
- calculate absolute weighted score
- calculate peer benchmarks
- calculate criterion gaps
- calculate relative percentages
- calculate PPI
- rank suppliers
- apply tie-break rules

Python will perform all arithmetic,
benchmarking and ranking.

For EVERY active criterion return:

1. criterion_id
2. score
3. justification
4. evidence

Scoring:

- Score from 0 to the criterion maximum.
- Use ONLY information contained in the proposal.
- Never invent evidence.
- Missing information should reduce the score.
- Evidence should be specific.
- Be objective and consistent.

ACTIVE CRITERIA
===============

{criteria_text}

SUPPLIER PROPOSAL
=================

{proposal_text}
"""

        result = (
            self.structured_llm.invoke(
                prompt
            )
        )

        return result


In [63]:
# ============================================================
# TOOL 2
# VALIDATION TOOL
# ============================================================

@tool
def validate_evaluation(
    llm_result: dict,
    criteria: list
) -> dict:
    """
    Validation Tool.

    Validates and normalizes LLM output.

    Handles:
    - missing criteria
    - malformed scores
    - scores below zero
    - scores above maximum
    - malformed evidence
    - malformed justification
    """

    results_by_id = {

        item["criterion_id"]: item

        for item in llm_result.get(
            "criteria",
            []
        )
    }

    validated = []

    for criterion in criteria:

        criterion_id = criterion[
            "criterion_id"
        ]

        max_score = criterion[
            "max_score"
        ]

        warnings = []

        result = results_by_id.get(
            criterion_id
        )

        # ----------------------------------------------------
        # Missing criterion
        # ----------------------------------------------------

        if result is None:

            validated.append({

                "criterion_id":
                    criterion_id,

                "raw_score":
                    None,

                "normalized_score":
                    0.0,

                "justification":
                    "No evaluation returned.",

                "evidence":
                    [],

                "warnings": [
                    "Missing criterion evaluation. "
                    "Score defaulted to 0."
                ]
            })

            continue

        # ----------------------------------------------------
        # Score
        # ----------------------------------------------------

        raw_score = result.get(
            "score"
        )

        try:

            normalized_score = float(
                raw_score
            )

        except (
            TypeError,
            ValueError
        ):

            normalized_score = 0.0

            warnings.append(
                "Malformed score. "
                "Defaulted to 0."
            )

        # ----------------------------------------------------
        # Lower bound
        # ----------------------------------------------------

        if normalized_score < 0:

            normalized_score = 0.0

            warnings.append(
                "Score below 0. "
                "Clipped to 0."
            )

        # ----------------------------------------------------
        # Upper bound
        # ----------------------------------------------------

        if normalized_score > max_score:

            normalized_score = max_score

            warnings.append(
                f"Score exceeded maximum "
                f"{max_score}. "
                f"Clipped to maximum."
            )

        # ----------------------------------------------------
        # Evidence
        # ----------------------------------------------------

        evidence = result.get(
            "evidence",
            []
        )

        if not isinstance(
            evidence,
            list
        ):

            evidence = [
                str(evidence)
            ]

            warnings.append(
                "Evidence was not a list. "
                "Converted to list."
            )

        # ----------------------------------------------------
        # Justification
        # ----------------------------------------------------

        justification = result.get(
            "justification",
            ""
        )

        if not isinstance(
            justification,
            str
        ):

            justification = str(
                justification
            )

            warnings.append(
                "Justification was not text."
            )

        validated.append({

            "criterion_id":
                criterion_id,

            "raw_score":
                raw_score,

            "normalized_score":
                normalized_score,

            "justification":
                justification,

            "evidence":
                evidence,

            "warnings":
                warnings
        })

    return {

        "criteria":
            validated
    }


In [64]:
# ============================================================
# DETERMINISTIC ABSOLUTE SCORE
# ============================================================

def calculate_absolute_score(
    validation_result,
    criteria
):

    results_by_id = {

        r["criterion_id"]: r

        for r in validation_result[
            "criteria"
        ]
    }

    absolute_score = 0.0

    criterion_results = []

    for criterion in criteria:

        criterion_id = criterion[
            "criterion_id"
        ]

        score = results_by_id[
            criterion_id
        ][
            "normalized_score"
        ]

        max_score = criterion[
            "max_score"
        ]

        weight = criterion[
            "weight"
        ]

        # ----------------------------------------------------
        # Formula:
        #
        # (criterion score / max score)
        # * criterion weight
        # * 100
        # ----------------------------------------------------

        weighted_score = (

            score / max_score

        ) * weight * 100

        absolute_score += (
            weighted_score
        )

        original = results_by_id[
            criterion_id
        ]

        criterion_results.append({

            "criterion_id":
                criterion_id,

            "criterion_name":
                criterion[
                    "name"
                ],

            "raw_score":
                original[
                    "raw_score"
                ],

            "score":
                score,

            "max_score":
                max_score,

            "weight":
                weight,

            "weighted_score":
                round(
                    weighted_score,
                    4
                ),

            "justification":
                original[
                    "justification"
                ],

            "evidence":
                original[
                    "evidence"
                ],

            "warnings":
                original[
                    "warnings"
                ]
        })

    return (
        round(
            absolute_score,
            4
        ),

        criterion_results
    )


In [65]:
# ============================================================
# TOOL 3
# RANKING TOOL
# ============================================================

@tool
def calculate_ranking(
    supplier_results: list,
    criteria: list
) -> list:
    """
    Ranking Tool.

    Completely deterministic Python.

    Performs:
        - peer benchmark
        - criterion gap
        - relative percentage
        - PPI
        - tie-break
        - final ranking

    NO LLM is used.
    """

    # ========================================================
    # STEP 1
    # CRITERION BENCHMARK
    # ========================================================

    benchmarks = {}

    for criterion in criteria:

        criterion_id = criterion[
            "criterion_id"
        ]

        max_score = criterion[
            "max_score"
        ]

        valid_scores = []

        for supplier in supplier_results:

            for result in supplier[
                "criteria"
            ]:

                if (
                    result[
                        "criterion_id"
                    ]
                    == criterion_id
                ):

                    score = result[
                        "score"
                    ]

                    if (
                        score is not None
                        and
                        0 <= score <= max_score
                    ):

                        valid_scores.append(
                            score
                        )

        if valid_scores:

            benchmarks[
                criterion_id
            ] = max(
                valid_scores
            )

        else:

            benchmarks[
                criterion_id
            ] = 0.0

    # ========================================================
    # STEP 2
    # PEER METRICS
    # ========================================================

    for supplier in supplier_results:

        ppi = 0.0

        for result in supplier[
            "criteria"
        ]:

            criterion_id = result[
                "criterion_id"
            ]

            score = result[
                "score"
            ]

            weight = result[
                "weight"
            ]

            benchmark = benchmarks[
                criterion_id
            ]

            # ------------------------------------------------
            # Relative performance
            # ------------------------------------------------

            if benchmark == 0:

                # Safe handling when all suppliers
                # have zero for the criterion.
                relative_percentage = 100.0

            else:

                relative_percentage = (

                    score / benchmark

                ) * 100

            # ------------------------------------------------
            # Criterion gap
            # ------------------------------------------------

            criterion_gap = (
                score - benchmark
            )

            # ------------------------------------------------
            # Store metrics
            # ------------------------------------------------

            result[
                "benchmark_score"
            ] = round(
                benchmark,
                4
            )

            result[
                "criterion_gap"
            ] = round(
                criterion_gap,
                4
            )

            result[
                "relative_percentage"
            ] = round(
                relative_percentage,
                4
            )

            # ------------------------------------------------
            # PPI
            # ------------------------------------------------

            ppi += (
                relative_percentage
                * weight
            )

        supplier[
            "ppi"
        ] = round(
            ppi,
            4
        )

    # ========================================================
    # STEP 3
    # DETERMINISTIC TIE BREAK
    #
    # 1. Higher PPI
    # 2. Earlier submission date
    # 3. Higher experience rating
    # 4. Supplier name ASC
    # ========================================================

    ranked = sorted(

        supplier_results,

        key=lambda supplier: (

            -supplier[
                "ppi"
            ],

            supplier[
                "submission_date"
            ],

            -supplier[
                "experience_rating"
            ],

            supplier[
                "supplier_name"
            ].lower()
        )
    )

    # ========================================================
    # STEP 4
    # ASSIGN RANK
    # ========================================================

    for rank, supplier in enumerate(
        ranked,
        start=1
    ):

        supplier[
            "final_rank"
        ] = rank

    return ranked


In [66]:
# ============================================================
# LANGGRAPH STATE
# ============================================================

class RFPState(TypedDict, total=False):

    rfp_run_id: int

    criteria: List[
        Dict[str, Any]
    ]

    suppliers: List[
        Dict[str, Any]
    ]

    all_results: List[
        Dict[str, Any]
    ]

    rankings: List[
        Dict[str, Any]
    ]


In [67]:
# ============================================================
# LANGGRAPH NODE
# LOAD CRITERIA
# ============================================================

def load_criteria_node(
    state: RFPState
):

    print(
        "\n[ORCHESTRATOR]"
        " Loading active criteria..."
    )

    criteria = load_active_criteria()

    validate_criteria(
        criteria
    )

    state[
        "criteria"
    ] = criteria

    print(
        f"Loaded {len(criteria)} active criteria."
    )

    return state


In [68]:
# ============================================================
# LANGGRAPH NODE
# CREATE RUN
# ============================================================

def create_run_node(
    state: RFPState
):

    print(
        "\n[ORCHESTRATOR]"
        " Creating RFP run..."
    )

    conn = get_connection()
    cursor = conn.cursor()

    created_at = (
        datetime.now().isoformat()
    )

    cursor.execute("""
        INSERT INTO rfp_runs
        (
            created_at,
            status
        )
        VALUES (?, ?)
    """, (

        created_at,

        "RUNNING"
    ))

    run_id = cursor.lastrowid

    # --------------------------------------------------------
    # Snapshot criteria
    # --------------------------------------------------------

    for criterion in state[
        "criteria"
    ]:

        cursor.execute("""
            INSERT INTO rfp_run_criteria
            (
                rfp_run_id,
                criterion_id,
                name,
                description,
                weight,
                max_score
            )
            VALUES (?, ?, ?, ?, ?, ?)
        """, (

            run_id,

            criterion[
                "criterion_id"
            ],

            criterion[
                "name"
            ],

            criterion[
                "description"
            ],

            criterion[
                "weight"
            ],

            criterion[
                "max_score"
            ]
        ))

    conn.commit()
    conn.close()

    state[
        "rfp_run_id"
    ] = run_id

    print(
        f"Created RFP Run ID: {run_id}"
    )

    return state


In [69]:
# ============================================================
# LANGGRAPH NODE
# EVALUATE SUPPLIERS
# ============================================================

def evaluate_suppliers_node(
    state: RFPState
):

    criteria = state[
        "criteria"
    ]

    suppliers = state[
        "suppliers"
    ]

    print(
        "\n[ORCHESTRATOR]"
        " Starting supplier evaluation..."
    )

    # --------------------------------------------------------
    # Instantiate Evaluation Agent
    # --------------------------------------------------------

    evaluation_agent = (
        EvaluationAgent(
            model_name=MODEL_NAME
        )
    )

    all_results = []

    # ========================================================
    # SUPPLIER LOOP
    # ========================================================

    for index, supplier in enumerate(
        suppliers,
        start=1
    ):

        supplier_name = supplier[
            "supplier_name"
        ]

        pdf_path = supplier[
            "pdf_path"
        ]

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"SUPPLIER {index}: "
            f"{supplier_name}"
        )

        print(
            "=" * 70
        )

        # ====================================================
        # TOOL 1
        # DOCUMENT TOOL
        # ====================================================

        print(
            "\n[DOCUMENT TOOL]"
            " Extracting PDF text..."
        )

        proposal_text = (
            extract_pdf_text.invoke(
                {
                    "pdf_path":
                        pdf_path
                }
            )
        )

        print(
            f"Extracted "
            f"{len(proposal_text)} characters."
        )

        # ====================================================
        # AGENT
        # EVALUATION AGENT
        # ====================================================

        print(
            "\n[EVALUATION AGENT]"
            " Asking LLM to evaluate proposal..."
        )

        llm_result = (
            evaluation_agent.evaluate(
                proposal_text,
                criteria
            )
        )

        llm_result_dict = (
            llm_result.model_dump()
        )

        print(
            "[EVALUATION AGENT]"
            " Evaluation received."
        )

        # ====================================================
        # TOOL 2
        # VALIDATION TOOL
        # ====================================================

        print(
            "\n[VALIDATION TOOL]"
            " Validating LLM response..."
        )

        validation_result = (
            validate_evaluation.invoke(
                {
                    "llm_result":
                        llm_result_dict,

                    "criteria":
                        criteria
                }
            )
        )

        print(
            "[VALIDATION TOOL]"
            " Validation complete."
        )

        # ====================================================
        # DETERMINISTIC SCORE
        # ====================================================

        absolute_score, criterion_results = (
            calculate_absolute_score(
                validation_result,
                criteria
            )
        )

        print(
            f"\nAbsolute Score: "
            f"{absolute_score:.2f}"
        )

        # ====================================================
        # CREATE SUPPLIER RESULT
        # ====================================================

        supplier_result = {

            "supplier_name":
                supplier_name,

            "submission_date":
                supplier[
                    "submission_date"
                ],

            "experience_rating":
                supplier[
                    "experience_rating"
                ],

            "absolute_score":
                absolute_score,

            "criteria":
                criterion_results
        }

        all_results.append(
            supplier_result
        )

    state[
        "all_results"
    ] = all_results

    return state


In [70]:
# ============================================================
# LANGGRAPH NODE
# RANK SUPPLIERS
# ============================================================

def rank_suppliers_node(
    state: RFPState
):

    print(
        "\n[ORCHESTRATOR]"
        " Calling Ranking Tool..."
    )

    rankings = (
        calculate_ranking.invoke(
            {
                "supplier_results":
                    state[
                        "all_results"
                    ],

                "criteria":
                    state[
                        "criteria"
                    ]
            }
        )
    )

    state[
        "rankings"
    ] = rankings

    print(
        "[RANKING TOOL]"
        " Ranking completed."
    )

    return state

In [71]:
# ============================================================
# LANGGRAPH NODE
# PERSIST RESULTS
# ============================================================

def persist_results_node(
    state: RFPState
):

    print(
        "\n[ORCHESTRATOR]"
        " Persisting results..."
    )

    run_id = state[
        "rfp_run_id"
    ]

    rankings = state[
        "rankings"
    ]

    conn = get_connection()
    cursor = conn.cursor()

    # ========================================================
    # SUPPLIERS
    # ========================================================

    for supplier in rankings:

        cursor.execute("""
            INSERT INTO rfp_suppliers
            (
                rfp_run_id,
                supplier_name,
                submission_date,
                experience_rating,
                absolute_score,
                ppi,
                final_rank,
                result_json
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (

            run_id,

            supplier[
                "supplier_name"
            ],

            supplier[
                "submission_date"
            ],

            supplier[
                "experience_rating"
            ],

            supplier[
                "absolute_score"
            ],

            supplier[
                "ppi"
            ],

            supplier[
                "final_rank"
            ],

            json.dumps(
                supplier,
                indent=2
            )
        ))

        supplier_id = (
            cursor.lastrowid
        )

        # ====================================================
        # CRITERION RESULTS
        # ====================================================

        for result in supplier[
            "criteria"
        ]:

            cursor.execute("""
                INSERT INTO evaluation_results
                (
                    supplier_id,
                    criterion_id,
                    raw_score,
                    normalized_score,
                    justification,
                    evidence_json,
                    warnings_json,
                    weighted_score,
                    benchmark_score,
                    criterion_gap,
                    relative_percentage
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (

                supplier_id,

                result[
                    "criterion_id"
                ],

                result[
                    "raw_score"
                ],

                result[
                    "score"
                ],

                result[
                    "justification"
                ],

                json.dumps(
                    result[
                        "evidence"
                    ]
                ),

                json.dumps(
                    result[
                        "warnings"
                    ]
                ),

                result[
                    "weighted_score"
                ],

                result.get(
                    "benchmark_score"
                ),

                result.get(
                    "criterion_gap"
                ),

                result.get(
                    "relative_percentage"
                )
            ))

    # ========================================================
    # COMPLETE RUN
    # ========================================================

    cursor.execute("""
        UPDATE rfp_runs

        SET status = ?

        WHERE rfp_run_id = ?
    """, (

        "COMPLETED",

        run_id
    ))

    conn.commit()
    conn.close()

    print(
        f"Results stored for Run ID {run_id}"
    )

    return state


In [72]:
# ============================================================
# BUILD LANGGRAPH
# ============================================================

def build_graph():

    graph = StateGraph(
        RFPState
    )

    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    graph.add_node(
        "load_criteria",
        load_criteria_node
    )

    graph.add_node(
        "create_run",
        create_run_node
    )

    graph.add_node(
        "evaluate_suppliers",
        evaluate_suppliers_node
    )

    graph.add_node(
        "rank_suppliers",
        rank_suppliers_node
    )

    graph.add_node(
        "persist_results",
        persist_results_node
    )

    # --------------------------------------------------------
    # Edges
    #
    # No conditional edges.
    # --------------------------------------------------------

    graph.add_edge(
        START,
        "load_criteria"
    )

    graph.add_edge(
        "load_criteria",
        "create_run"
    )

    graph.add_edge(
        "create_run",
        "evaluate_suppliers"
    )

    graph.add_edge(
        "evaluate_suppliers",
        "rank_suppliers"
    )

    graph.add_edge(
        "rank_suppliers",
        "persist_results"
    )

    graph.add_edge(
        "persist_results",
        END
    )

    return graph.compile()


In [73]:
# ============================================================
# DISPLAY RESULTS
# ============================================================

def display_results(final_state):

    rankings = final_state[
        "rankings"
    ]

    print("\n")
    print("=" * 90)
    print("FINAL RFP LEADERBOARD")
    print("=" * 90)

    print(
        f"{'Rank':<8}"
        f"{'Supplier':<25}"
        f"{'Absolute':<15}"
        f"{'PPI':<15}"
        f"{'Experience':<12}"
    )

    print("-" * 90)

    for supplier in rankings:

        print(

            f"{supplier['final_rank']:<8}"

            f"{supplier['supplier_name']:<25}"

            f"{supplier['absolute_score']:<15.2f}"

            f"{supplier['ppi']:<15.2f}"

            f"{supplier['experience_rating']:<12.1f}"
        )

    print("=" * 90)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    for supplier in rankings:

        print("\n")
        print(
            f"SUPPLIER: "
            f"{supplier['supplier_name']}"
        )

        print(
            f"Final Rank: "
            f"{supplier['final_rank']}"
        )

        print(
            f"Absolute Score: "
            f"{supplier['absolute_score']:.2f}"
        )

        print(
            f"PPI: "
            f"{supplier['ppi']:.2f}%"
        )

        print("-" * 90)

        for result in supplier[
            "criteria"
        ]:

            print(
                f"\nCriterion: "
                f"{result['criterion_name']}"
            )

            print(
                f"Score: "
                f"{result['score']:.2f} / "
                f"{result['max_score']:.2f}"
            )

            print(
                f"Weighted Score: "
                f"{result['weighted_score']:.2f}"
            )

            print(
                f"Benchmark: "
                f"{result.get('benchmark_score', 0):.2f}"
            )

            print(
                f"Gap: "
                f"{result.get('criterion_gap', 0):.2f}"
            )

            print(
                f"Relative Performance: "
                f"{result.get('relative_percentage', 0):.2f}%"
            )

            print(
                "Justification:"
            )

            print(
                result[
                    "justification"
                ]
            )

            print(
                "Evidence:"
            )

            for evidence in result[
                "evidence"
            ]:

                print(
                    f"  - {evidence}"
                )

            if result[
                "warnings"
            ]:

                print(
                    "Warnings:"
                )

                for warning in result[
                    "warnings"
                ]:

                    print(
                        f"  - {warning}"
                    )


In [74]:
# ============================================================
# MAIN EXECUTION FUNCTION
# ============================================================

def run_rfp_evaluation(
    suppliers
):

    print(
        "=" * 90
    )

    print(
        "RFP EVALUATION SYSTEM"
    )

    print(
        "LangGraph + Tools + Evaluation Agent"
    )

    print(
        "=" * 90
    )

    # --------------------------------------------------------
    # Initialize DB
    # --------------------------------------------------------

    initialize_database()

    # --------------------------------------------------------
    # Build graph
    # --------------------------------------------------------

    app = build_graph()

    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    initial_state: RFPState = {

        "suppliers":
            suppliers
    }

    # --------------------------------------------------------
    # Execute LangGraph
    # --------------------------------------------------------

    final_state = app.invoke(
        initial_state
    )

    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    display_results(
        final_state
    )

    return final_state

In [75]:
from google.colab import files

uploaded = files.upload()

Saving nexaworks.pdf to nexaworks (3).pdf
Saving brightpath_tech.pdf to brightpath_tech (3).pdf
Saving apex_systems.pdf to apex_systems (3).pdf
Saving orbit_digital.pdf to orbit_digital (3).pdf


In [76]:
suppliers = [

    {
        "supplier_name":
            "Apex Systems",

        "submission_date":
            "2026-08-25",

        "experience_rating":
            8.5,

        "pdf_path":
            "apex_systems.pdf"
    },

    {
        "supplier_name":
            "BrightPath Tech",

        "submission_date":
            "2026-08-24",

        "experience_rating":
            6.5,

        "pdf_path":
            "brightpath_tech.pdf"
    },

    {
        "supplier_name":
            "NexaWorks",

        "submission_date":
            "2026-08-26",

        "experience_rating":
            9.0,

        "pdf_path":
            "nexaworks.pdf"
    },

    {
        "supplier_name":
            "Orbit Digital",

        "submission_date":
            "2026-08-23",

        "experience_rating":
            9.5,

        "pdf_path":
            "orbit_digital.pdf"
    }
]

In [77]:
final_state = run_rfp_evaluation(
    suppliers
)

RFP EVALUATION SYSTEM
LangGraph + Tools + Evaluation Agent
Existing evaluation criteria preserved.

[ORCHESTRATOR] Loading active criteria...
Loaded 5 active criteria.

[ORCHESTRATOR] Creating RFP run...
Created RFP Run ID: 7

[ORCHESTRATOR] Starting supplier evaluation...

SUPPLIER 1: Apex Systems

[DOCUMENT TOOL] Extracting PDF text...
Extracted 4623 characters.

[EVALUATION AGENT] Asking LLM to evaluate proposal...
[EVALUATION AGENT] Evaluation received.

[VALIDATION TOOL] Validating LLM response...
[VALIDATION TOOL] Validation complete.

Absolute Score: 88.00

SUPPLIER 2: BrightPath Tech

[DOCUMENT TOOL] Extracting PDF text...
Extracted 3474 characters.

[EVALUATION AGENT] Asking LLM to evaluate proposal...
[EVALUATION AGENT] Evaluation received.

[VALIDATION TOOL] Validating LLM response...
[VALIDATION TOOL] Validation complete.

Absolute Score: 68.00

SUPPLIER 3: NexaWorks

[DOCUMENT TOOL] Extracting PDF text...
Extracted 4519 characters.

[EVALUATION AGENT] Asking LLM to evaluat